In [11]:
import librosa
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import warnings

In [12]:
warnings.filterwarnings('ignore')

In [13]:
AUDIO_PATH = '../data/ESC-50-master/ESC-50-master/audio/'
META_PATH = '../data/ESC-50-master/ESC-50-master/meta/esc50.csv'

df = pd.read_csv(META_PATH)
print(df.shape)
print(df.head())

(2000, 7)
            filename  fold  target        category  esc10  src_file take
0   1-100032-A-0.wav     1       0             dog   True    100032    A
1  1-100038-A-14.wav     1      14  chirping_birds  False    100038    A
2  1-100210-A-36.wav     1      36  vacuum_cleaner  False    100210    A
3  1-100210-B-36.wav     1      36  vacuum_cleaner  False    100210    B
4  1-101296-A-19.wav     1      19    thunderstorm  False    101296    A


In [14]:
def extract_mfcc(file_path, n_mfcc=13):
    y, sr = librosa.load(file_path, sr=None)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    return np.mean(mfccs, axis=1)

features = []
labels = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    file_path = os.path.join(AUDIO_PATH, row['filename'])
    mfcc = extract_mfcc(file_path)
    features.append(mfcc)
    labels.append(row['category'])

X = np.array(features)
y = np.array(labels)

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

100%|██████████| 2000/2000 [01:08<00:00, 29.34it/s]

X shape: (2000, 13)
y shape: (2000,)


In [15]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f'Train size: {X_train.shape}')
print(f'Test size: {X_test.shape}')
print(f'Classes: {len(le.classes_)}')

Train size: (1600, 13)
Test size: (400, 13)
Classes: 50


In [16]:
svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm.fit(X_train, y_train)

y_pred = svm.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.4f}')
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 0.5175
                  precision    recall  f1-score   support

        airplane       0.86      0.75      0.80         8
       breathing       0.33      0.25      0.29         8
  brushing_teeth       0.44      0.50      0.47         8
     can_opening       0.57      0.50      0.53         8
        car_horn       0.44      0.50      0.47         8
             cat       0.42      0.62      0.50         8
        chainsaw       0.29      0.25      0.27         8
  chirping_birds       0.56      0.62      0.59         8
    church_bells       0.50      0.38      0.43         8
        clapping       0.83      0.62      0.71         8
     clock_alarm       0.50      0.75      0.60         8
      clock_tick       0.14      0.12      0.13         8
        coughing       0.22      0.25      0.24         8
             cow       0.67      0.50      0.57         8
  crackling_fire       0.50      0.38      0.43         8
        crickets       0.35      0.75      0.48       

In [17]:
def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=None)

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfccs, axis=1)
    mfcc_std = np.std(mfccs, axis=1)

    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = np.mean(chroma, axis=1)

    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    centroid_mean = np.mean(spectral_centroid)

    zcr = librosa.feature.zero_crossing_rate(y)
    zcr_mean = np.mean(zcr)

    return np.hstack([mfcc_mean, mfcc_std, chroma_mean, centroid_mean, zcr_mean])

features2 = []
for idx, row in tqdm(df.iterrows(), total=len(df)):
    file_path = os.path.join(AUDIO_PATH, row['filename'])
    feat = extract_features(file_path)
    features2.append(feat)

X2 = np.array(features2)
print(f'X2 shape: {X2.shape}')

100%|██████████| 2000/2000 [02:16<00:00, 14.64it/s]

X2 shape: (2000, 40)


In [18]:
X2_scaled = scaler.fit_transform(X2)

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

svm2 = SVC(kernel='rbf', C=10, gamma='scale', random_state=42, probability=True)
svm2.fit(X2_train, y2_train)

y2_pred = svm2.predict(X2_test)
acc2 = accuracy_score(y2_test, y2_pred)
print(f'Accuracy: {acc2:.4f}')

Accuracy: 0.6475


In [19]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X2_train, y2_train)
rf_pred = rf.predict(X2_test)
rf_acc = accuracy_score(y2_test, rf_pred)

xgb = XGBClassifier(n_estimators=200, random_state=42, eval_metric='mlogloss')
xgb.fit(X2_train, y2_train)
xgb_pred = xgb.predict(X2_test)
xgb_acc = accuracy_score(y2_test, xgb_pred)

print(f'SVM accuracy:          {acc2:.4f}')
print(f'Random Forest accuracy: {rf_acc:.4f}')
print(f'XGBoost accuracy:       {xgb_acc:.4f}')

SVM accuracy:          0.6475
Random Forest accuracy: 0.5675
XGBoost accuracy:       0.5325


In [20]:
import joblib
import os

os.makedirs('../models', exist_ok=True)

joblib.dump(svm2, '../models/svm_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(le, '../models/label_encoder.pkl')

print('SVM model saved')
print('Scaler saved')
print('Label encoder saved')

SVM model saved
Scaler saved
Label encoder saved
